In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
q1 = 'I just discovered the course, can I still join?'
q2 = 'I just found out about the program, can I still enroll?'

In [10]:
v1 = model.encode(q1)

In [11]:
v2 = model.encode(q2)

In [12]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [8]:
v1.dot(dv)

np.float32(0.39572883)

In [9]:
v2.dot(dv)

np.float32(0.463655)

### Embedding datasets

In [1]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-23 09:49:27--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 888 [text/plain]
Saving to: ‘ingest.py’

ingest.py           100%[===================>]     888  --.-KB/s    in 0s      

2026-06-23 09:49:27 (41.5 MB/s) - ‘ingest.py’ saved [888/888]



In [2]:
from ingest import load_faq_data

documents = load_faq_data()

In [ ]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [6]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [7]:
vectors[10].shape

(384,)

In [13]:
v1.dot(vectors[10])

np.float32(0.3134635)

In [ ]:
# Slow version
scores = []

for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

In [ ]:
# Faster:
import numpy as np
X = np.array(vectors)

scores = X.dot(v1)

In [ ]:
# Finding the document with the highest cosine similarity (i.e. answer to the question)
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(538), np.float32(0.8317791))

In [ ]:
documents[538]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [ ]:
#  For top 5 results
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1] # instead of this manual reversal can also just do in one line with: np.argsort(-scores)[:5]
print(top5)
scores[top5]

[538 907 625   2 503]


array([0.8317791 , 0.6845695 , 0.61755615, 0.60880876, 0.58479667],
      dtype=float32)

In [24]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.8317791
{'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.', 'doc_id': '74eb249bbf'}

0.6845695
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.', 'doc_id': '41aabbd7c5'}

0.61755615
{'course': 'mlops-zoomcamp', 'section': 'Ge

### Replacing manual vector embedding with libraries

In [25]:
from minsearch import VectorSearch
vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [27]:
vindex.search(v1, num_results=5, filter_dict={'course': 'llm-zoomcamp'})

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offere

### Integrating into a RAG agent

In [28]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-23 10:49:49--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-23 10:49:49 (49.3 MB/s) - ‘rag_helper.py’ saved [2134/2134]



In [29]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [30]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [31]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [ ]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

In [32]:
# Override the RAG helper with RAGVector (vector search capabilities)
class RAGVector(RAGBase):
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [34]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [35]:
query = 'I just found out about the program, can I still sign up?'
vector_assistant.rag(query)

'Yes, you can still join. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.'

### Problems with this brute-force in-memory approach:
- It rebuilds the index on every startup
- It keeps everything in memory
- It searches by brute force


In [36]:
# Solving this with sqlitesearch (in-disk db) by bringing in persistent memory for embeddings and it uses approximate nearest neighbours (ANN) to find relevant vector search space, instead of more expensive exact nearest neighbours (NN) approach

In [37]:
from sqlitesearch import VectorSearchIndex
vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf', # This is ANN
    db_path='faq_vectors2.db'
)

In [38]:
vs_index.fit(vectors, documents)

In [ ]:
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vs_index.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

In [40]:
vs_index.close()